# Cell embedding computation across Fig. 1 representation routes

The workflow follows the feature-extraction and cell-embedding routes summarized in Fig. 1a and the model definitions in Table 2 and the Supplementary Methods. Repository wrappers are used without changing their algorithms. The main example assumes a cell-by-gene AnnData object containing raw UMI counts and unique cell identifiers.

The workflow has three branches: direct transcriptomic models, static gene or protein representations pooled into cells, and expression baselines. Protein sequence and GO-text vectors are prepared with `human-protein-representation-workflow.ipynb` before they are mapped to genes and pooled here.

## 1. Model coverage

The 25 benchmark entries are covered below. scNET and totalVI are included as additional, dataset-restricted routes described in the Supplementary Methods.

| Family | Models or baselines | Cell representation route |
|---|---|---|
| Expression and scRNA-seq | Raw expression, HVG, one_hot_v, scVI, scFoundation, STATE, Geneformer, Gene2vec | direct matrix, trained or pretrained transcriptomic model, or gene-vector pooling |
| Protein sequence | ESM2-15B, ESM2-3B, ESM2-650M, ESM2-8M, ESM2-CLS, ESM2-SEG0, ESM2-TM-FT, ESM2-SubLoc-FT, ESMC, PAIR, xTrimoPGLM-3B, che_evo | HGNC mapping followed by cell-specific top-gene pooling |
| Functional text | Qwen, GPT-small, GPT-large, GenePT-ada, GenePT-GPT-large | GO-text protein vectors with top-gene pooling, or GenePT expression weighting |
| Restricted additions | scNET, totalVI | dataset-trained graph embedding, or paired RNA/ADT latent embedding |

ESM2-650M denotes mean pooling unless the readout is named explicitly. ESM2-15B is the 5,120-dimensional gene-indexed protein prior supplied with STATE, not the 2,058-dimensional STATE cell embedding.

## 2. Environments and external assets

Use separate environments for model families whose upstream dependencies conflict. The shared analysis environment is suitable for AnnData inspection and pooling of precomputed vectors. scVI/totalVI, scFoundation, STATE, Geneformer and scNET require their own upstream packages.

Required external assets include the scFoundation checkpoint and 19,264-gene index, STATE SE-600M, Geneformer `gf-12L-30M-i2048`, and the upstream scNET package and human PPI resources. Protein checkpoints and generated protein matrices are described in `../docs/data-and-model-resources.md`. Large matrices and model weights are not stored in this repository.

### Reference environment commands

Create the environments independently. Pin upstream revisions after confirming them for the target CUDA driver; do not merge all model families into the shared environment.

In [ ]:
environment_commands = {
    "shared pooling and validation": [
        "conda create -n cell-embedding python=3.11 -y",
        "conda run -n cell-embedding python -m pip install -r requirements-core.txt",
    ],
    "scVI and totalVI": [
        "conda create -n scvi python=3.10 -y",
        "conda run -n scvi python -m pip install scvi-tools scanpy anndata",
    ],
    "Geneformer v1 compatibility": [
        "conda create -n geneformer python=3.10 -y",
        "conda run -n geneformer python -m pip install transformers==4.21.3 datasets==2.5.2 scanpy==1.9.1",
    ],
}
for environment, commands in environment_commands.items():
    print(f"\n# {environment}")
    print("\n".join(commands))

### Checkpoint and source retrieval

Install Git LFS and the Hugging Face CLI before downloading model repositories. SE-600M and Geneformer can be retrieved from their official Hugging Face records. Obtain the scFoundation checkpoint and 19,264-gene index through the upstream scFoundation release instructions. Clone STATE, scFoundation and scNET only when their local wrappers are used, and record the selected commit. Protein-model retrieval is covered by the protein-representation notebook.

In [ ]:
asset_commands = [
    "huggingface-cli download arcinstitute/SE-600M --local-dir models/SE-600M",
    "huggingface-cli download ctheodoris/Geneformer --include gf-12L-30M-i2048/* --local-dir models/Geneformer",
    "git clone https://github.com/ArcInstitute/state models/state",
    "git clone https://github.com/biomap-research/scFoundation models/scFoundation-source",
    "git clone https://github.com/madilabcode/scNET models/scNET-source",
]
print("\n".join(asset_commands))

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import numpy as np
import anndata as ad

ROOT = Path.cwd().resolve()
if ROOT.name == "tutorials":
    ROOT = ROOT.parent

MODULE = ROOT / "cell-embedding-construction-across-representation-models"
FOUNDATION = MODULE / "scripts/foundation-models"
PROTEIN_TO_CELL = MODULE / "scripts/protein-to-cell"
SHARED = ROOT / "agent-based-pg-tfidf-analysis-framework/skills/cell-cluster-marker-tfidf-analysis"
DATA_UTIL = ROOT / "data_util"

RNA_H5AD = Path("inputs/raw_counts.h5ad")
ADT_H5AD = Path("inputs/adt_counts.h5ad")
OUTPUT_ROOT = Path("outputs/cell-embeddings")
HGNC = DATA_UTIL / "mappings/hgnc_complete_set.txt"
GENE2VEC = DATA_UTIL / "embeddings/gene2vec/gene2vec_dim_200_iter_9.npz"

def command_text(parts):
    text_parts = [str(part) for part in parts]
    return subprocess.list2cmdline(text_parts) if os.name == "nt" else shlex.join(text_parts)


## 3. Inspect the RNA input before model-specific preprocessing

Rows must be cells and columns must be genes. Preserve `obs_names` as the authoritative cell order. Equal row counts between a matrix and an embedding file do not prove correct alignment. The checks below inspect the input without changing it.

In [ ]:
adata = ad.read_h5ad(RNA_H5AD, backed="r")
assert adata.n_obs > 0 and adata.n_vars > 0
assert adata.obs_names.is_unique, "Cell identifiers must be unique."
assert adata.var_names.is_unique, "Gene identifiers must be unique before model-specific mapping."
print("shape:", adata.shape)
print("first cells:", adata.obs_names[:3].tolist())
print("first genes:", adata.var_names[:3].tolist())
adata.file.close()

## 4. Build the shared HVG and Raw baselines

The baseline wrapper selects 3,000 HVGs on a CP10K-plus-log1p copy, writes the normalized HVG matrix as `cell_embedding_hvg.npy`, and preserves pre-normalization counts for those genes in `HVG_gene_count.h5ad`. `--save_raw` also writes the all-gene normalized matrix as `cell_embedding_raw.npy`. The HVG count object is the manuscript input for the static top-100 pooling routes.

In [ ]:
baseline_out = OUTPUT_ROOT / "baseline"
baseline_command = [
    "python", FOUNDATION / "baseline_embedding_inference.py",
    "--input_path", RNA_H5AD,
    "--output_path", baseline_out,
    "--n_top_genes", 3000,
    "--enable_cp10k", "True",
    "--enable_log1p", "True",
    "--seed", 42,
    "--save_raw",
]
print(command_text(baseline_command))
HVG_COUNTS = baseline_out / "HVG_gene_count.h5ad"

### one_hot_v

The public one-hot wrapper selects the top 100 ranked genes in each cell, filters them through the HGNC and available-protein mapping used by the supplied embedding directory, and builds a dataset-specific binary vocabulary. It does not encode amino-acid residues. The embedding directory is used only to determine eligible mapped genes.

In [ ]:
ELIGIBILITY_EMBEDDING_DIR = Path("inputs/protein-embeddings/esm2-650m-mean")
one_hot_command = [
    "python", PROTEIN_TO_CELL / "top_gene_cell_embedding_one_hot_v.py",
    "--hgnc_file", HGNC,
    "--h5ad_file", HVG_COUNTS,
    "--embedding_dir", ELIGIBILITY_EMBEDDING_DIR,
    "--id_column", "ID",
    "--save_path", OUTPUT_ROOT / "one_hot_v",
    "--top_k", 100,
    "--max_proteins", 10,
    "--seed", 42,
]
print(command_text(one_hot_command))

## 5. Direct transcriptomic cell embeddings

These routes read a transcriptomic matrix and return cell embeddings directly. Use raw UMI counts unless the upstream model requires a separate conversion step.

### scVI

The wrapper filters cells with fewer than 200 detected genes and genes present in fewer than three cells, trains a ZINB scVI model, and exports a 40-dimensional latent matrix. Dataset90_2 and Dataset90_4 used `--skip_qc`; do not apply that exception to other datasets without checking cell loss.

In [ ]:
scvi_command = [
    "python", FOUNDATION / "scvi_feature_extraction.py",
    "--open_path", RNA_H5AD,
    "--save_path", OUTPUT_ROOT / "scvi",
    "--n_latent", 40,
    "--max_epochs", 20,
    "--batch_size", 256,
    "--n_hidden", 128,
    "--n_layers", 1,
    "--device", "cuda:0",
]
print(command_text(scvi_command))

### scFoundation

The preprocessing step aligns the matrix to 19,264 genes. Cell inference uses `pool_type=all`, `version=rde` and effective `tgthighres=f1`. The output concatenates two special-token vectors, gene-wise maximum pooling and gene-wise mean pooling, giving 3,072 features.

In [ ]:
SCFOUNDATION_ROOT = Path("models/scFoundation")
SCFOUNDATION_CHECKPOINT = Path("models/scFoundation/checkpoint.ckpt")
SCFOUNDATION_INDEX = Path("models/scFoundation/OS_scRNA_gene_index.19264.tsv")
scf_pre_out = OUTPUT_ROOT / "scfoundation-preprocess"
scf_preprocess = [
    "python", FOUNDATION / "scRNAseq_h5ad_preprocessing_under_scfoundation_sc_ATD2.py",
    "--open_path", RNA_H5AD,
    "--save_path", scf_pre_out,
    "--sparse_matrix", "True",
]
SCFOUNDATION_PREPROCESSED = scf_pre_out / ("preprocessed_all_" + RNA_H5AD.name)
scf_inference = [
    "python", FOUNDATION / "get_embedding_sc_ATD2.py",
    "--task_name", "dataset",
    "--input_type", "singlecell",
    "--output_type", "cell",
    "--pool_type", "all",
    "--data_path", SCFOUNDATION_PREPROCESSED,
    "--pre_normalized", "F",
    "--version", "rde",
    "--save_path", OUTPUT_ROOT / "scfoundation",
    "--tgthighres", "f1",
    "--cuda_device", 0,
    "--model_path", SCFOUNDATION_CHECKPOINT,
    "--OS_scRNA_gene_index_path", SCFOUNDATION_INDEX,
]
print(command_text(scf_preprocess))
print(command_text(scf_inference))

### STATE

The selected SE-600M route aligns gene symbols to the model vocabulary and returns 2,058 features: 2,048 semantic cell-state dimensions followed by 10 dataset or batch dimensions. This output is different from the 5,120-dimensional STATE-supplied gene vectors used by the ESM2-15B static route.

In [ ]:
STATE_ROOT = Path("models/state")
STATE_MODEL = Path("models/SE-600M")
state_command = [
    "python", FOUNDATION / "cell_embedding_inference.py",
    "--open_path", RNA_H5AD,
    "--save_path", OUTPUT_ROOT / "state/state",
    "--model_path", STATE_MODEL,
    "--root_dir", STATE_ROOT,
    "--batch_size", 1,
    "--device", "cuda:0",
]
print(command_text(state_command))

### Geneformer

Tokenization maps genes to the Geneformer identifier space and ranks normalized expression values. The selected `gf-12L-30M-i2048` run truncates each cell to 2,048 tokens and produces a 512-dimensional embedding. The wrapper passes `emb_layer=-1` to the installed Geneformer extractor; confirm the exact layer semantics against the pinned upstream version.

In [ ]:
GENEFORMER_ASSETS = Path("models/Geneformer/gf-12L-30M-i2048")
GENEFORMER_MAP = Path("models/Geneformer/gene-symbol-to-ensembl.csv")
geneformer_tokenize = [
    "python", FOUNDATION / "token_geneformer_shift.py",
    "--open_path", RNA_H5AD.parent,
    "--save_path", OUTPUT_ROOT / "geneformer-tokenized",
    "--open_path_conference_data", GENEFORMER_ASSETS,
    "--mapping_file", GENEFORMER_MAP,
    "--G38_convert", "False",
]
geneformer_extract = [
    "python", FOUNDATION / "Embedding.py",
    "--open_path", OUTPUT_ROOT / "geneformer-tokenized",
    "--save_path", OUTPUT_ROOT / "geneformer-dataset",
    "--save_path_embedding", OUTPUT_ROOT / "geneformer",
    "--open_path_conference_data", GENEFORMER_ASSETS,
    "--device_choose", "cuda:0",
    "--trunc_num", 2048,
    "--batch_size", 2,
]
print(command_text(geneformer_tokenize))
print(command_text(geneformer_extract))

### scNET: restricted additional route

The study evaluated scNET only on GSE223779 and Dataset81. The wrapper uses scNET internal preprocessing, human mode, 25 neighbours, five epochs, cell splitting and 32 batches. It trains on the supplied dataset and exports `embedded_cells.npy`; it is not one of the 25 models evaluated across every collection.

In [ ]:
scnet_command = [
    "python", FOUNDATION / "scnet/feature_extraction.py",
    "--open_path", RNA_H5AD,
    "--save_path", OUTPUT_ROOT / "scnet",
    "--max_epoch", 5,
    "--split_cells",
    "--number_of_batches", 32,
]
print(command_text(scnet_command))

## 6. Static gene and protein representations pooled into cells

The manuscript route first creates an HVG count object, ranks genes within each cell, keeps the top 100 positions, maps genes to available vectors and averages the matched vectors. In the public top-gene script, ranking uses `argsort` over the supplied row and does not explicitly remove zeros. When fewer than 100 entries are nonzero, zero-valued genes can therefore enter the ranked set.

The public top-gene protein wrapper uses the first mapped UniProt identifier for each gene. The separate generic mapper can average multiple protein entries per gene, but it then constructs cells by expression-weighted matrix multiplication rather than per-cell mean pooling. Keep these implementations distinct when reproducing an experiment.

### Protein sequence, GO-text and che_evo matrices

Prepare one protein-indexed matrix and its aligned identifier table for each route. The same pooling command applies to ESM2-3B, ESM2-650M mean, ESM2-8M, ESM2-CLS, ESM2-SEG0, ESM2-TM-FT, ESM2-SubLoc-FT, ESMC, PAIR, xTrimoPGLM-3B, Qwen, GPT-small, GPT-large and che_evo. ESM2-15B uses the gene-indexed STATE file in the next section.

The public repository contains extraction workflows for ESM2-650M, ESMC, PAIR, xTrimoPGLM and GO text. Other ESM2 sizes and the two LoRA-adapted matrices require their matching checkpoint and extraction record. The public repository does not contain a standalone che_evo generator; supply the prepared 62-dimensional protein table described in the Supplementary Methods rather than substituting a different descriptor.

In [ ]:
protein_routes = {
    "ESM2-3B": Path("inputs/protein-embeddings/esm2-3b-mean"),
    "ESM2-650M": Path("inputs/protein-embeddings/esm2-650m-mean"),
    "ESM2-8M": Path("inputs/protein-embeddings/esm2-8m-mean"),
    "ESM2-CLS": Path("inputs/protein-embeddings/esm2-650m-cls"),
    "ESM2-SEG0": Path("inputs/protein-embeddings/esm2-650m-seg0"),
    "ESM2-TM-FT": Path("inputs/protein-embeddings/esm2-tm-ft"),
    "ESM2-SubLoc-FT": Path("inputs/protein-embeddings/esm2-subloc-ft"),
    "ESMC": Path("inputs/protein-embeddings/esmc-600m-mean"),
    "PAIR": Path("inputs/protein-embeddings/pair-mean"),
    "xTrimoPGLM-3B": Path("inputs/protein-embeddings/xtrimopglm-3b-mean"),
    "Qwen": Path("inputs/protein-embeddings/go-qwen"),
    "GPT-small": Path("inputs/protein-embeddings/go-gpt-small"),
    "GPT-large": Path("inputs/protein-embeddings/go-gpt-large"),
    "che_evo": Path("inputs/protein-embeddings/che-evo-62d"),
}

for model_name, embedding_dir in protein_routes.items():
    command = [
        "python", SHARED / "top_gene_cell_embedding.py",
        "--hgnc_file", HGNC,
        "--h5ad_file", HVG_COUNTS,
        "--embedding_dir", embedding_dir,
        "--id_column", "ID",
        "--save_path", OUTPUT_ROOT / model_name,
        "--top_k", 100,
        "--max_proteins", 10,
        "--seed", 42,
    ]
    print(f"\n# {model_name}")
    print(command_text(command))

### ESM2-15B from the STATE protein prior

Export `protein_embeddings.pt` from SE-600M to `STATE_tok_5120_Emb.h5ad` with the protein-representation tutorial. Rows are gene symbols and columns are 5,120 protein features. The cell pooling wrapper matches those gene symbols directly and writes `top_gene_cell_embedding.npy`.

In [ ]:
STATE_GENE_EMBEDDING = Path("inputs/gene-embeddings/STATE_tok_5120_Emb.h5ad")
state_15b_command = [
    "python", PROTEIN_TO_CELL / "top_gene_cell_embedding_STATE_v.py",
    "--embedding_dir", STATE_GENE_EMBEDDING,
    "--h5ad_file", HVG_COUNTS,
    "--save_path", OUTPUT_ROOT / "ESM2-15B",
    "--top_k", 100,
    "--seed", 42,
]
print(command_text(state_15b_command))

### Gene2vec

Gene2vec uses the bundled 200-dimensional gene vectors. The wrapper selects the top 100 ranked genes per cell, mean-pools matched vectors and assigns a zero vector when no selected gene is present in the Gene2vec vocabulary.

In [ ]:
gene2vec_command = [
    "python", PROTEIN_TO_CELL / "top_gene_cell_embedding_gene2vec.py",
    "--embedding_dir", GENE2VEC,
    "--h5ad_file", HVG_COUNTS,
    "--save_path", OUTPUT_ROOT / "gene2vec",
    "--top_k", 100,
    "--seed", 42,
]
print(command_text(gene2vec_command))

### GenePT-ada and GenePT-GPT-large

Use the released GenePT dictionaries generated from NCBI gene summaries. The original GenePT-w construction used in this study computes the expression-weighted sum `C = X @ E`, where `X` is the CP10K-plus-log1p cell-by-gene matrix and `E` is the aligned gene-by-feature matrix. The public wrapper implements this formulation directly and does not divide each row by its total expression weight. It should therefore be described as the original GenePT-w expression-weighted sum, not as a normalized weighted average or as top-100 mean pooling. The main GenePT-w comparison uses all matched genes; omit `--top_k`.

In [ ]:
genept_routes = {
    "GenePT-ada": Path("inputs/gene-embeddings/genept-ada.pkl"),
    "GenePT-GPT-large": Path("inputs/gene-embeddings/genept-gpt-large.pkl"),
}
for model_name, genept_pickle in genept_routes.items():
    command = [
        "python", PROTEIN_TO_CELL / "gene_to_protein_embedding_mapper_Genept_v.py",
        "--genept_pickle", genept_pickle,
        "--h5ad_file", RNA_H5AD,
        "--save_path", OUTPUT_ROOT / model_name,
        "--seed", 42,
    ]
    print(f"\n# {model_name}")
    print(command_text(command))

## 7. totalVI for paired RNA and ADT

totalVI is a paired multimodal reference rather than an ADT-only embedding. The wrapper intersects cell identifiers, restores a shared sorted order, trains on RNA counts plus `obsm['protein_counts']`, and exports a 40-dimensional joint latent matrix. Use `--skip_qc` only for the recorded Dataset90_2 and Dataset90_4 exceptions or for the tiny schema example.

In [ ]:
totalvi_command = [
    "python", FOUNDATION / "totalvi_combine_feature_extraction.py",
    "--rna_path", RNA_H5AD,
    "--adt_path", ADT_H5AD,
    "--save_path", OUTPUT_ROOT / "totalvi",
    "--n_latent", 40,
    "--max_epochs", 20,
    "--batch_size", 128,
    "--n_hidden", 256,
    "--device", "cuda:0",
]
print(command_text(totalvi_command))

## 8. Validate every output before benchmarking

Keep an ordered cell-ID file beside each NumPy matrix. Check finite values, expected dimensions, coverage statistics and exact row alignment. A zero vector can indicate failed gene matching rather than a biological state.

In [ ]:
def validate_embedding(matrix_path, expected_cell_ids, cell_id_path=None, expected_dim=None):
    matrix = np.load(matrix_path, mmap_mode="r")
    assert matrix.ndim == 2
    assert matrix.shape[0] == len(expected_cell_ids)
    if expected_dim is not None:
        assert matrix.shape[1] == expected_dim
    assert np.isfinite(matrix).all()
    if cell_id_path is not None:
        observed = Path(cell_id_path).read_text(encoding="utf-8").splitlines()
        assert observed == list(map(str, expected_cell_ids))
    zero_rows = int(np.count_nonzero(np.linalg.norm(matrix, axis=1) == 0))
    return {"shape": tuple(matrix.shape), "zero_rows": zero_rows}

source = ad.read_h5ad(RNA_H5AD, backed="r")
source_cells = source.obs_names.astype(str).tolist()
source.file.close()

# Example after scVI completes:
# validate_embedding(OUTPUT_ROOT / "scvi/embedded_cells.npy", source_cells, expected_dim=40)

## 9. Expected primary outputs

| Route | Primary file | Expected feature dimension |
|---|---|---:|
| Raw | `baseline/cell_embedding_raw.npy` | number of retained genes |
| HVG | `baseline/cell_embedding_hvg.npy` | 3,000 in the full setting |
| one_hot_v | `one_hot_v/top_gene_one_hot_cell_embedding.npy` | dataset-specific vocabulary size |
| scVI | `scvi/embedded_cells.npy` | 40 |
| scFoundation | generated task/checkpoint filename | 3,072 |
| STATE | `state/state_embeddings.npy` | 2,058 |
| Geneformer v1 | extractor output | 512 |
| Gene2vec | `gene2vec/top_gene_cell_embedding.npy` | 200 |
| ESM2-15B static | `ESM2-15B/top_gene_cell_embedding.npy` | 5,120 |
| Other protein or GO routes | `<model>/top_gene_cell_embedding.npy` | source-vector dimension |
| GenePT | `<model>/genept_cell_embedding.npy` | 1,536 or 3,072 for the named releases |
| totalVI | `totalvi/embedded_cells.npy` | 40 |
| scNET | model-named subdirectory containing `embedded_cells.npy` | upstream model output |

Record the input h5ad, ordered cell IDs, gene or protein vector source, checkpoint revision, pooling field, output shape and mapping coverage with each completed run.

## 10. Recommended run order

1. Validate raw counts and cell identifiers.
2. Build the baseline outputs and `HVG_gene_count.h5ad`.
3. Prepare or download each gene/protein representation with its ordered identifier table.
4. Run direct transcriptomic models in their pinned environments.
5. Run static top-gene pooling from the same HVG count object.
6. Run totalVI only when matched RNA and ADT counts are available; run scNET only for its documented study datasets unless extending the benchmark.
7. Validate row order, dimensions, finite values, zero rows and mapping coverage before clustering or supervised evaluation.